In [2]:
run_path = "./results/20260101_203526"

In [3]:
import pandas as pd
import matplotlib.pyplot as plt

# 读取 CSV 文件
result_df = pd.read_csv(run_path + '/result.csv')
result_diff_df = pd.read_csv(run_path + '/result_diff.csv')

print("result.csv:")
print(result_df.head())
print("\nresult_diff.csv:")
print(result_diff_df.head())

result.csv:
            Unnamed: 0 time_usage  use_mc  custom_mc  num_rows  num_cols  \
0  2026-01-01 20:35:44   00:00:17   False      False        16         8   
1  2026-01-01 20:35:55   00:00:11   False      False        16         8   
2  2026-01-01 20:37:27   00:01:32   False      False        32        16   
3  2026-01-01 20:37:47   00:00:19   False      False        32        16   
4  2026-01-01 20:43:18   00:05:31   False      False        64        32   

   gen_unused_cells          r_delay           r_pavg           r_pstc  \
0              True   [4.327391e-10]   [-3.85507e-05]  [-9.201274e-07]   
1             False  [4.3166693e-10]  [-3.863338e-05]   [-9.38733e-07]   
2              True   [5.711278e-10]    [-0.00011507]  [-2.466761e-06]   
3             False  [5.7086267e-10]    [-0.00011564]   [-2.44618e-06]   
4              True  [8.0201797e-10]    [-0.00038989]  [-7.908124e-06]   

          r_pdyn          w_delay           w_pavg           w_pstc  \
0  [-0.00010537

In [12]:
# 整合数据
result_df['type'] = 'actual'
result_diff_df['type'] = 'diff'

# 合并 DataFrame
combined_df = pd.concat([result_df, result_diff_df], ignore_index=True)

# 转换时间列为 datetime
combined_df.iloc[:, 0] = pd.to_datetime(combined_df.iloc[:, 0])

# 解析数值列
import ast
numeric_cols = ['r_delay', 'r_pavg', 'r_pstc', 'r_pdyn', 'w_delay', 'w_pavg', 'w_pstc', 'w_pdyn']
for col in numeric_cols:
    combined_df[col] = combined_df[col].apply(lambda x: ast.literal_eval(x)[0] if isinstance(x, str) and x.startswith('[') else x)

# 重构表格：行是参数，列是 num_rows x num_cols，下分 gen_unused_cells
combined_df.loc[combined_df['type'] == 'diff', 'gen_unused_cells'] = 'diff'
actual_df = combined_df  # 现在包括 diff
melted = actual_df.melt(id_vars=['num_rows', 'num_cols', 'gen_unused_cells'], value_vars=numeric_cols)
pivot_table = melted.pivot_table(index='variable', columns=['num_rows', 'num_cols', 'gen_unused_cells'], values='value')

# 设置列为 MultiIndex
pivot_table.columns.names = ['num_rows', 'num_cols', 'type']

formatted_df = pivot_table.apply(lambda col: col.apply(lambda x: f"{x*100:.2f}%" if col.name[2] == 'diff' else f"{x:.4e}"), axis=0)

print("重构后的表格:")
display(formatted_df)

重构后的表格:


/tmp/ipykernel_2450731/384977480.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'diff' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  combined_df.loc[combined_df['type'] == 'diff', 'gen_unused_cells'] = 'diff'


num_rows           16                                32                       \
num_cols           8                                 16                        
type            False         True    diff        False         True    diff   
variable                                                                       
r_delay    4.3167e-10   4.3274e-10  -0.25%   5.7086e-10   5.7113e-10  -0.05%   
r_pavg    -3.8633e-05  -3.8551e-05   0.21%  -1.1564e-04  -1.1507e-04   0.50%   
r_pdyn    -1.0555e-04  -1.0537e-04   0.17%  -3.1695e-04  -3.1528e-04   0.53%   
r_pstc    -9.3873e-07  -9.2013e-07   2.02%  -2.4462e-06  -2.4668e-06  -0.83%   
w_delay    3.8028e-10   3.7908e-10   0.32%   3.5771e-10   3.5761e-10   0.03%   
w_pavg    -2.6537e-05  -2.8861e-05  -8.05%  -7.2897e-05  -7.3835e-05  -1.27%   
w_pdyn    -7.2248e-05  -7.8591e-05  -8.07%  -1.9780e-04  -2.0037e-04  -1.28%   
w_pstc    -7.3424e-07  -7.9242e-07  -7.34%  -2.2549e-06  -2.2727e-06  -0.79%   

num_rows           64                       
num_cols           32                       
type            False         True    diff  
variable                                    
r_delay    8.0237e-10   8.0202e-10   0.04%  
r_pavg    -3.9287e-04  -3.8989e-04   0.76%  
r_pdyn    -1.0781e-03  -1.0696e-03   0.80%  
r_pstc    -7.8414e-06  -7.9081e-06  -0.84%  
w_delay    3.7409e-10   3.7426e-10  -0.05%  
w_pavg    -2.3698e-04  -2.3567e-04   0.56%  
w_pdyn    -6.4332e-04  -6.4056e-04   0.43%  
w_pstc    -7.2213e-06  -6.9006e-06   4.65%

In [23]:
# 转换 time_usage 为秒（仅对 actual 数据）
def time_to_seconds(time_str):
    h, m, s = map(int, time_str.split(':'))
    return h * 3600 + m * 60 + s

combined_df.loc[combined_df['type'] == 'actual', 'time_usage_seconds'] = combined_df.loc[combined_df['type'] == 'actual', 'time_usage'].apply(time_to_seconds)

# 画折线图：(num_rows,num_cols) vs time_usage_seconds，按 gen_unused_cells 分组
actual_df = combined_df[combined_df['type'] == 'actual']
actual_df = actual_df.sort_values(['num_rows', 'num_cols'])
true_df = actual_df[actual_df['gen_unused_cells'] == True]
false_df = actual_df[actual_df['gen_unused_cells'] == False]

x_labels = [f"({r},{c})" for r, c in zip(true_df['num_rows'], true_df['num_cols'])]

# plt.figure(figsize=(10, 6))
plt.plot(x_labels, true_df['time_usage_seconds'], marker='o', label='before acceleration')
plt.plot(x_labels, false_df['time_usage_seconds'], marker='o', label='after acceleration')
plt.xlabel('(num_rows, num_cols)')
plt.ylabel('Time Usage (seconds)')
# plt.title('Time Usage by (num_rows, num_cols) and gen_unused_cells')
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
# ./images
plt.savefig('./images/time_usage_comparison.svg')
plt.close()
# plt.show()